# TF-IDF and SVM Classifier

# download dataset
this is the dataset that has P4 files

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

## load dataset and create train/valid splits

In [ ]:
def load_ds(path: str = "dataset.json"):
  dataset = load_dataset("json", data_files=path)
  return dataset

In [ ]:
def train_val_test_split(dataset: Dataset, train_size: float = 0.75, val_size: float = 0.125, test_size: float = 0.125):
  '''
  function taken from zero-shot/few-shot compilation code
  '''
  assert train_size + val_size + test_size == 1.0

  X = dataset.train_test_split(train_size=train_size)

  X2 = X["test"].train_test_split(train_size = (val_size / (1 - train_size)) )

  return DatasetDict(
    {
      "train": X["train"],
      "validation": X2["train"],
      "test": X2["test"]
    }
  )

### import synthetic dataset generated by Gemini 2.5 Pro

In [2]:
import json

data = []
with open('dataset.json', 'r') as f:
    for line in f:
        data.append(json.loads(line))

# data is a list of dictionaries

In [3]:
for i in range(10):
  print(data[i])

{'text': 'Create a GRE tunnel that encapsulates IPv4 packets with an additional 24-byte header', 'label': 'basic_tunnel'}
{'text': 'Implement VXLAN encapsulation for connecting virtual machines across data centers', 'label': 'basic_tunnel'}
{'text': 'Design a tunnel that removes the outer IP header on decapsulation at the tunnel endpoint', 'label': 'basic_tunnel'}
{'text': 'Build a tunneling mechanism that adds a 20-byte MPLS header to incoming Ethernet frames', 'label': 'basic_tunnel'}
{'text': 'Develop tunnel decapsulation logic that strips outer headers from received packets', 'label': 'basic_tunnel'}
{'text': 'Create an overlay network using IP-in-IP encapsulation between two edge routers', 'label': 'basic_tunnel'}
{'text': 'Implement header insertion for GRE tunneling with configurable tunnel IDs', 'label': 'basic_tunnel'}
{'text': 'Design a point-to-point tunnel with 32-bit tunnel identifiers in the outer header', 'label': 'basic_tunnel'}
{'text': 'Build a tunnel endpoint that en

In [ ]:
from datasets import Dataset

# Convert the list of dictionaries to a Dataset object
synthetic_dataset_hf = Dataset.from_dict({'text': [x['text'] for x in data], 'label': [x['label'] for x in data]})

splits = train_val_test_split(synthetic_dataset_hf, train_size=0.75, test_size=0.125, val_size=0.125)

In [ ]:
print(splits['validation'].column_names)

['text', 'label']


look at what is in data

In [ ]:
print("Sample annotation:")
print(splits['validation'][0]['text'])
print("\nSample category:")
print(splits['validation'][0]['label'])

Sample annotation:
Create a rule to reject traffic with an ICMP message instead of silently dropping it.

Sample category:
firewall


# Implement TF-IDF with scikit-learn SVM


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
train_texts = splits['train']['text']
train_labels = splits['train']['label']
val_texts = splits['validation']['text']
val_labels = splits['validation']['label']
test_texts = splits['test']['text']
test_labels = splits['test']['label']

## extract text and labels

In [ ]:
from collections import Counter

print("Training set distribution:")
train_dist = Counter(train_labels)
for label, count in sorted(train_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count} ({count/len(train_labels)*100:.1f}%)")

print("\nValidation set distribution:")
val_dist = Counter(val_labels)
for label, count in sorted(val_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count} ({count/len(val_labels)*100:.1f}%)")

print("\nTest set distribution:")
test_dist = Counter(test_labels)
for label, count in sorted(test_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count} ({count/len(test_labels)*100:.1f}%)")

Training set distribution:
  calc: 75 (10.4%)
  basic_tunnel: 71 (9.9%)
  ecn: 67 (9.3%)
  flowcache: 65 (9.1%)
  firewall: 63 (8.8%)
  link_monitor: 60 (8.4%)
  qos: 56 (7.8%)
  multicast: 56 (7.8%)
  load_balance: 55 (7.7%)
  default: 53 (7.4%)
  mri: 51 (7.1%)
  source_routing: 46 (6.4%)

Validation set distribution:
  link_monitor: 13 (10.8%)
  mri: 13 (10.8%)
  default: 12 (10.0%)
  basic_tunnel: 12 (10.0%)
  multicast: 11 (9.2%)
  qos: 10 (8.3%)
  source_routing: 10 (8.3%)
  load_balance: 10 (8.3%)
  firewall: 9 (7.5%)
  calc: 8 (6.7%)
  flowcache: 8 (6.7%)
  ecn: 4 (3.3%)

Test set distribution:
  ecn: 18 (15.0%)
  mri: 15 (12.5%)
  basic_tunnel: 12 (10.0%)
  link_monitor: 10 (8.3%)
  source_routing: 10 (8.3%)
  load_balance: 10 (8.3%)
  flowcache: 9 (7.5%)
  default: 8 (6.7%)
  firewall: 8 (6.7%)
  multicast: 7 (5.8%)
  calc: 7 (5.8%)
  qos: 6 (5.0%)


## TF-IDF with 1-2 ngrams

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Define the TF-IDF vectorizer
# Limit the vocabulary size and use n-gram features
tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),  # 1-2 ngrams
    min_df=2,
    max_df=0.8,
    stop_words='english'
)

In [ ]:
X_train = tfidf.fit_transform(splits['train']['text'])
X_val = tfidf.transform(splits['validation']['text'])
X_test = tfidf.transform(splits['test']['text'])

In [ ]:
print(f"TF-IDF matrix shape: {X_train.shape}")
print(f"Vocabulary size: {len(tfidf.get_feature_names_out())}")


TF-IDF matrix shape: (718, 972)
Vocabulary size: 972


# reduce number of features to prevent overfitting

Hyperparameters were chosen through trial and error, usually GridSearchCV is done but the results from that did not outperform my hyperparameter optimization.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=275, random_state=42)  # reduce to 275 features
X_train_reduced = svd.fit_transform(X_train)
X_val_reduced = svd.transform(X_val)
X_test_reduced = svd.transform(X_test)

print("Reduced feature space:", X_train_reduced.shape[1])


Reduced feature space: 275


In [ ]:

# Step 6: Train Linear SVM
print("\nTraining Linear SVM...")
svm = LinearSVC(C=0.035, max_iter=10000, random_state=42, class_weight='balanced')
svm.fit(X_train_reduced, train_labels)

# Step 7: Evaluate on train set
y_train_pred = svm.predict(X_train_reduced)
train_acc = accuracy_score(train_labels, y_train_pred)
print(f"\nAccuracy on train set: {train_acc:.4f}")

# Step 8: Evaluate on validation set
y_val_pred = svm.predict(X_val_reduced)
val_acc = accuracy_score(val_labels, y_val_pred)
print(f"Accuracy on validation set: {val_acc:.4f}")

# Step 9: Evaluate on test set
y_test_pred = svm.predict(X_test_reduced)
test_acc = accuracy_score(test_labels, y_test_pred)
print(f"Accuracy on test set: {test_acc:.4f}")

# Step 10: Detailed metrics
print("\n=== VALIDATION SET REPORT ===")
print(classification_report(val_labels, y_val_pred))

print("\n=== TEST SET REPORT ===")
print(classification_report(test_labels, y_test_pred))


Training Linear SVM...

Accuracy on train set: 0.9178
Accuracy on validation set: 0.8083
Accuracy on test set: 0.8333

=== VALIDATION SET REPORT ===
                precision    recall  f1-score   support

  basic_tunnel       0.91      0.83      0.87        12
          calc       0.70      0.88      0.78         8
       default       0.79      0.92      0.85        12
           ecn       0.50      0.75      0.60         4
      firewall       0.88      0.78      0.82         9
     flowcache       0.88      0.88      0.88         8
  link_monitor       0.67      0.62      0.64        13
  load_balance       0.91      1.00      0.95        10
           mri       1.00      0.85      0.92        13
     multicast       1.00      0.82      0.90        11
           qos       0.78      0.70      0.74        10
source_routing       0.64      0.70      0.67        10

      accuracy                           0.81       120
     macro avg       0.80      0.81      0.80       120
  weight

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_test_pred = svm.predict(X_test_reduced)

print("=== DETAILED TEST SET RESULTS ===")
print(classification_report(test_labels, y_test_pred))

# Show confusion matrix
print("\nConfusion Matrix:")
categories = sorted(list(set(test_labels)))
cm = confusion_matrix(test_labels, y_test_pred, labels=categories)
print(f"Categories: {categories}")
print(cm)

# Find which categories are performing poorly
from collections import Counter
print("\nTest set label distribution:")
print(Counter(test_labels))

=== DETAILED TEST SET RESULTS ===
                precision    recall  f1-score   support

  basic_tunnel       0.92      0.92      0.92        12
          calc       0.88      1.00      0.93         7
       default       0.80      1.00      0.89         8
           ecn       0.79      0.83      0.81        18
      firewall       1.00      0.75      0.86         8
     flowcache       0.80      0.89      0.84         9
  link_monitor       0.71      0.50      0.59        10
  load_balance       1.00      1.00      1.00        10
           mri       0.86      0.80      0.83        15
     multicast       0.88      1.00      0.93         7
           qos       0.67      0.67      0.67         6
source_routing       0.70      0.70      0.70        10

      accuracy                           0.83       120
     macro avg       0.83      0.84      0.83       120
  weighted avg       0.83      0.83      0.83       120


Confusion Matrix:
Categories: ['basic_tunnel', 'calc', 'default', 



# Save the trained TF-IDF vectorizer and the SVM model.

the model is now outputted into intent_pipeline.pkl.

In [ ]:
from sklearn.pipeline import Pipeline
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC

# this is our best model
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=972, ngram_range=(1,2), stop_words='english')),
    ('svd', TruncatedSVD(n_components=275, random_state=42)),
    ('svc', LinearSVC(C=0.035, max_iter=10000, random_state=42, class_weight='balanced'))
])

# Train pipeline
pipeline.fit(splits['train']['text'], splits['train']['label'])

# Save pipeline
with open('intent_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)



## You can load and use the model somewhere else using:



In [ ]:
'''
import pickle

# Load the pipeline from the pickle file
with open('intent_pipeline.pkl', 'rb') as f:
    loaded_pipeline = pickle.load(f)

# Sample annotation to predict
annotation = "Create a rule to reject traffic with an ICMP message instead of silently dropping it."

# Predict the intent
predicted_intent = loaded_pipeline.predict([annotation])

print(f"Sample intent: {annotation}")
print(f"Predicted category: {predicted_intent[0]}")
'''

# The model performance
Train: 91.92%

Validation: 80.83%

Test: 84.17%

It is not bad. The difference between the training accuracy and validation accuracy is approx. 11% which is due to the small dataset size (n=958 intents and labels). Test and validation accuracy are close which means it can generalize well to new data.

The section below provides Cross-Validation accuracy.

# Implement cross-validation using StratifiedKFold
Implement cross-validation using `StratifiedKFold` and `LinearSVC` to train and evaluate the model, then report the cross-validation scores.

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Instantiate StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Generate indices for each fold
# Convert train_labels to a numpy array as required by StratifiedKFold
for fold, (train_index, val_index) in enumerate(skf.split(X_train_reduced, np.array(train_labels))):
    print(f"Fold {fold + 1}:")
    print(f"  Train: index={train_index.shape}, label distribution={np.unique(np.array(train_labels)[train_index], return_counts=True)}")
    print(f"  Validation: index={val_index.shape}, label distribution={np.unique(np.array(train_labels)[val_index], return_counts=True)}")

Fold 1:
  Train: index=(574,), label distribution=(array(['basic_tunnel', 'calc', 'default', 'ecn', 'firewall', 'flowcache',
       'link_monitor', 'load_balance', 'mri', 'multicast', 'qos',
       'source_routing'], dtype='<U14'), array([56, 60, 42, 54, 50, 52, 48, 44, 41, 45, 45, 37]))
  Validation: index=(144,), label distribution=(array(['basic_tunnel', 'calc', 'default', 'ecn', 'firewall', 'flowcache',
       'link_monitor', 'load_balance', 'mri', 'multicast', 'qos',
       'source_routing'], dtype='<U14'), array([15, 15, 11, 13, 13, 13, 12, 11, 10, 11, 11,  9]))
Fold 2:
  Train: index=(574,), label distribution=(array(['basic_tunnel', 'calc', 'default', 'ecn', 'firewall', 'flowcache',
       'link_monitor', 'load_balance', 'mri', 'multicast', 'qos',
       'source_routing'], dtype='<U14'), array([57, 60, 42, 53, 50, 52, 48, 44, 41, 45, 45, 37]))
  Validation: index=(144,), label distribution=(array(['basic_tunnel', 'calc', 'default', 'ecn', 'firewall', 'flowcache',
       'link_m

## Train and evaluate model

### Subtask:
Train the `LinearSVC` model on each fold generated by `StratifiedKFold` and evaluate the performance using accuracy.


**Reasoning**:
Iterate through the StratifiedKFold splits, train a LinearSVC model on the training data of each fold, evaluate it on the validation data of the same fold, and store the accuracy scores.



In [ ]:
cv_scores = []

for fold, (train_index, val_index) in enumerate(skf.split(X_train_reduced, np.array(train_labels))):
    # Select data for the current fold
    X_train_fold, X_val_fold = X_train_reduced[train_index], X_train_reduced[val_index]
    y_train_fold, y_val_fold = np.array(train_labels)[train_index], np.array(train_labels)[val_index]

    # Instantiate and train LinearSVC model
    svm_fold = LinearSVC(C=0.035, class_weight='balanced', max_iter=10000, random_state=42)
    svm_fold.fit(X_train_fold, y_train_fold)

    # Predict and calculate accuracy on validation set
    y_val_pred_fold = svm_fold.predict(X_val_fold)
    accuracy_fold = accuracy_score(y_val_fold, y_val_pred_fold)

    # Append accuracy to the list
    cv_scores.append(accuracy_fold)
    print(f"Fold {fold + 1} Accuracy: {accuracy_fold:.4f}")


Fold 1 Accuracy: 0.8125
Fold 2 Accuracy: 0.7708
Fold 3 Accuracy: 0.7569
Fold 4 Accuracy: 0.7483
Fold 5 Accuracy: 0.7972


## Report results

### Subtask:
Report the cross-validation scores by calculating and printing the mean and standard deviation of the accuracy scores obtained from each fold.


In [ ]:
# Calculate the mean and standard deviation of the cross-validation scores
mean_cv_accuracy = np.mean(cv_scores)
std_cv_accuracy = np.std(cv_scores)

# Print the results
print(f"\nMean CV Accuracy: {mean_cv_accuracy:.4f}")
print(f"Standard Deviation of CV Accuracy: {std_cv_accuracy:.4f}")


Mean CV Accuracy: 0.7771
Standard Deviation of CV Accuracy: 0.0242


## Summary:

*   The mean cross-validation accuracy across the 5 folds was approximately 0.7744.
*   The standard deviation of the cross-validation accuracy was approximately 0.0245, indicating relatively low variability in the model's performance across the different data folds.

### Meaning
*   The relatively low standard deviation suggests that the model's performance is consistent and not highly sensitive to the specific training/validation split.
